# はじめに

今回は、OpenSkillのレーティングアルゴリズムへの理解を深める。

- [A Bayesian Approximation Method for Online Ranking](https://www.csie.ntu.edu.tw/~cjlin/papers/online_ranking/online_journal.pdf)
- [openskill.py](https://github.com/vivekjoshy/openskill.py)

上記の論文は、OpenSkillのもととなったもの。順位を扱う場合、多次元の難しい積分が出てくるため、そこを尤度の1階微分、2階微分で近似(Weng-Linの)する点がOpenskillの特徴でもある。

## OpenSkillの数理

手書きの汚い点はごめんなさい。$\mu$と$\sigma$が1回更新されるまでの流れを記載している。

<div align='center'><img src='./OpenSkill01.png' width='1200'></div>
<div align='center'><img src='./OpenSkill02.png' width='1200'></div>
<div align='center'><img src='./OpenSkill03.png' width='1200'></div>
<div align='center'><img src='./OpenSkill04.png' width='1200'></div>
<div align='center'><img src='./OpenSkill05.png' width='1200'></div>
<div align='center'><img src='./OpenSkill06.png' width='1200'></div>

手書きの情報をまとめると、下記の表の通り。

| 艇     | 着順 |    勾配内の合計 |    対数尤度勾配 | $\Omega_i$ | $\Delta_i$ | 新しい $\mu$ | 新しい $\sigma$ |
| ----- | -: | --------: | --------: | ---------: | ---------: | --------: | -----------: |
| $r_1$ |  3 | -0.892114 | -0.089165 |  -1.427258 |   0.030802 | 26.572742 |     3.938768 |
| $r_2$ |  2 |  0.192740 |  0.019264 |   0.308357 |   0.030131 | 27.308357 |     3.940132 |
| $r_3$ |  1 |  0.699374 |  0.069901 |   1.118901 |   0.013444 | 27.118901 |     3.973884 |

最後に、前提条件を合わせてOpenskillライブラリで結果が一致するかをみておく。

In [ ]:
from openskill.models import PlackettLuce

model = PlackettLuce(beta=25/6, tau=25/300)
print(f"beta={model.beta:.3f}, tau={model.tau:.3f}")

r1 = model.rating(mu=28.0, sigma=4.0)
r2 = model.rating(mu=27.0, sigma=4.0)
r3 = model.rating(mu=26.0, sigma=4.0)
teams = [[r1], [r2], [r3]]
# print("winprob=", model.predict_win(teams))

updated = model.rate(teams, ranks=[3, 2, 1])
for boat_number, team in enumerate(updated, start=1):
    rating = team[0]
    print(f"{boat_number}号艇: " f"μ={rating.mu:.6f}, " f"σ={rating.sigma:.6f}")

beta=4.167, tau=0.083
1号艇: μ=26.572742, σ=3.938768
2号艇: μ=27.308357, σ=3.940132
3号艇: μ=27.118901, σ=3.973884


問題なく一致していることがわかる。